## STEP 0 
Import Libraries, Connect to Snowflake, & Initialize NLP Pipelines

In [ ]:
## TODO TODO
## 1. Make a robust, immune-to-timeout solution for querying
## 2. Figure out if you need some kind of `while true... except StopIteration` pattern like Gemini said
## 3. Finish the 5-24 to 5-31 chunk of NER analysis you still need to do
## 4. Get the merge-components of the sentiment analysis working
## 5. The functions to iterate through source data and writeback in batches is not DRY enough-- fix that

from datasets import Dataset
from datetime import datetime, timedelta
import os
import re
import snowflake.connector
from snowflake.connector.pandas_tools import write_pandas
from time import sleep
import torch
from transformers import pipeline

# this basically means "smoke em if you got em" where the "em" is NVIDIA GPU
DEVICE = 0 if torch.cuda.is_available() else -1

SF_USR = os.getenv('SF_USR')
SF_ID  = os.getenv('SF_ID')
SF_WH  = os.getenv('SF_WH')
SF_DB  = os.getenv('SF_DB')
SF_SC  = os.getenv('SF_SC')
SF_RL  = os.getenv('SF_RL')

# connect to database and init a cursor for querying
xct_params = {
    "user":                 SF_USR
   ,"account":              SF_ID
   ,"warehouse":            SF_WH
   ,"database":             SF_DB
   ,"schema":               SF_SC
   ,"role":                 SF_RL
   ,"private_key_file":     os.getenv('PRIVATE_KEY_PATH')
   ,"private_key_file_pwd": os.getenv('PRIVATE_KEY_PASSPHRASE')
   ,"authenticator":        os.getenv('SF_AUTH')
}

SF_XCT = snowflake.connector.connect(**xct_params)
CSR = SF_XCT.cursor()

# sentiment analyzer doo-dad instantiation
PIPL_SNT = pipeline(
    "sentiment-analysis",
    model="cardiffnlp/twitter-roberta-base-sentiment",
    tokenizer="cardiffnlp/twitter-roberta-base-sentiment",
    device=DEVICE,
    truncation=True,
    max_length = 512 
)
## This sentiment pipeline returns labels like ['LABEL_0', 'LABEL_1', 'LABEL_2']
## instead of ['Negative', 'Neutral', 'Positive']
## The below-linked mapping indicates which model-labels match to which
## human-understandable terms. 
## for verification, see this link:
##      https://huggingface.co/cardiffnlp/twitter-roberta-base-sentiment

# named-entity recognition doo-dad instantiation
PIPL_NER = pipeline(
    "ner",
    model="dslim/bert-base-NER",
    tokenizer="dslim/bert-base-NER",
    aggregation_strategy="simple",
    device=DEVICE,
    batch_size=256 
)

def execute_query(query: str
                 ,conn: snowflake.connector.connection.SnowflakeConnection = SF_XCT
                 ,cursor: snowflake.connector.cursor.SnowflakeCursor = CSR
                 ,connection_parameters: dict = xct_params
                 ,retry_attempts: int = 5
                 ) -> snowflake.connector.cursor.SnowflakeCursor:
    """
    Execute a query in such a way that it never fails unless Snowflake really really
    won't let you connect after multiple retries

    Args:
        query: The query you want to execute in Snowflake
        conn:  The Python Connection to Snowflake
        cursor: The cursor you use to execute queries in Snowflake via the connection
        connection_parameters: A dict containing the username, authentication, etc that lets you into Snowflake
        retry_attempts: Total number of times this function will reattempt to connect and query before giving 
                        up.

    """
    try:
        cursor.execute(query)
    except snowflake.connector.errors.Error as e:
        print(f"Connection to Snowflake went bad due to error: {e}\nAttempting to re-establish connection...")
        last_error_msg = e

        for i in range(retry_attempts):
            try:
                if conn:
                    conn.close()
                conn = snowflake.connector.connect(**connection_parameters)
                cursor = conn.cursor()
                cursor.execute(query)
                print(f"Successfully re-established connection on attempt {i}")
                return cursor
            except Exception as e:
                print(f"Reconnection Attempt {i} Failed: {e}")
                last_error_msg = e
                if i < retry_attempts: # only sleep & try again if you still have retries left
                    sleep(2 ** i) #increase time between each reconnection attempt exponentially-- this maxes out at just over 1 min of total 'waiting to retry' time for 5 attempts
        else: # i didn't know you could do `for... else` in python! cool!!
            print(f"Failed to re-establish connection and execute query after {retry_attempts} attempts. Aborting.")
            # Print the final error and then re-raise it
            raise last_error_msg 
  
def writeback_batch(source_table_query: str
                   ,source_table_name
                   ,target_table_name
                   ,target_columns: list
                   ,nlp_params: dict
                   ,source_table_filter: str = None
                   ,source_database: str = SF_DB
                   ,source_schema: str = SF_SC
                   ,target_database: str = SF_DB
                   ,target_schema: str = SF_SC
                   ,conn: snowflake.connector.connection.SnowflakeConnection = SF_XCT
                   ,cursor: snowflake.connector.cursor.SnowflakeCursor = CSR
                   ,connection_parameters: dict = xct_params):
    """
    This query is used for all the NLP workflows on this project. It works like this:

        1. Query some source table. Retrieve the results in batches
        2. For each batch, do some local NLP analysis
        3. Write the batch back to a target table
        4. Repeat until all batches are exhausted, printing detailed progress 
           reports along the way
    
    Args:
        source_table_query: The SQL query string used to retrieve the source data 
        source_table_name: The name of the source table in Snowflake
        source_table_filter: A SQL fragment that filters the source table. This
                             is input separate so it can be used with the part
                             where we count rows in the source table; don't want
                             an inaccurate count.
        target_table_name: The name of the table to which we are writing NLP data
        target_columns: The column-set of the target table (string list)
        nlp_params: A dict that indicates what NLP metric is being calculated,
                    what transformer pipeline is being used, and what the name
                    is of the column containing the text that is being analyzed.
        source_database: the database of the source table
        source_database: the schema of the source table
        target_database: the database of the target table
        target_database: the schema of the target table
        conn: Snowflake connection
        cursor: Snowflake connection's cursor
        connection_parameters: All details used to instantiate a Snowflake 
                               connection              
    """

    # need to be able to report progress during execution because this runs for so long
    # track some measures to help do that
    count_all_query = f"select count(*) from {source_database}.{source_schema}.{source_table_name}"
    if source_table_filter:
        count_all_query += f' {source_table_filter}'
    cursor = execute_query(count_all_query
                          ,conn=conn
                          ,cursor=cursor
                          ,connection_parameters=connection_parameters
                          )
    total_rows_in_source = cursor.fetchone()[0]
    c = 0
    rows_processed = 0
    pcnt_progress = 0
    
    # retrieve source data
    cursor = execute_query(source_table_query
                          ,conn=conn
                          ,cursor=cursor
                          ,connection_parameters=connection_parameters
                          )
    process_started_at = datetime.now()
    print(f"Initiated Named-Entity Recognition (NER) analysis at\n{process_started_at}")
    print(f"Downloading {(total_rows_in_source):,} total rows from {source_table_name.upper()}...\n\n")

    for batch in cursor.fetch_pandas_batches():
        batch_started_at = datetime.now()
        c              += 1
        rows_processed += 1
        pcnt_progress  += (len(batch)/total_rows_in_source) * 100
        print(f"\n{len(batch):,} rows downloaded from batch {(c):,}")

        # using this thing as input is more efficient than using `batch` directly for whatever reason
        batch_dataset = Dataset.from_pandas(batch[[nlp_params['target_text_colname']]], preserve_index=False)
        # execute NER analysis 
        nlp_output = PIPL_NER(batch_dataset[nlp_params['target_text_colname']])
        batch[nlp_params['nlp_metric'].upper()] = nlp_output
        
        # NER is called first-- if that's what we're doing, then fill in an empty sentana column for now-- we'll get it later
        if nlp_params['nlp_metric'].upper() == 'NER_ANALYSIS' and 'SENTIMENT_ANALYSIS' not in batch.columns:
            batch['SENTIMENT_ANALYSIS'] = None
        
        # match col order before writing
        batch = batch[target_columns]
        write_pandas(SF_XCT
                    ,batch
                    ,table_name        = target_table_name
                    ,database          = target_database
                    ,schema            = target_schema
                    ,use_logical_type  = True
                    ,auto_create_table = False
                    ,overwrite         = False
            )
        
        #### more prints to show velocity 
        # ... of this specific batch
        batch_finished_at = datetime.now()
        print(f"Batch {(c):,} completed at {batch_finished_at}")
        print(f"{len(batch):,} rows from batch written to {target_table_name} ({(pcnt_progress):,.1f}% of source rows processed)")
        total_seconds_for_batch = (batch_finished_at - batch_started_at).total_seconds
        minute_time_for_batch = total_seconds_for_batch // 60
        second_time_for_batch = total_seconds_for_batch % 60
        print(f"Batch Process Time = {minute_time_for_batch}min {second_time_for_batch}s")
       
        # ...of the entire process
        min_elapsed_since_process_start = (datetime.now() - process_started_at).total_seconds/60
        processing_velocity = round(rows_processed/min_elapsed_since_process_start, 2)
        print(f"Processing rate = {processing_velocity} rows/minute")
    
    process_finished_at = datetime.now()
    total_seconds_for_process = (process_finished_at - process_started_at).total_seconds
    minute_time_for_process = total_seconds_for_process // 60
    second_time_for_process = total_seconds_for_process % 60
    print(f"\n{(c):,} batches totaling {(total_rows_in_source):,} rows were processed in {minute_time_for_process}min {second_time_for_process}s")
    print(f"Average process velocity is {round((total_rows_in_source/(total_seconds_for_process/60)), 1):,.1f}")

# DRIVER
if __name__ == "__main__":
    ## NER ANALYSIS
    query = f"""
    select a.*
    from bluesky_db.main.firehose_processed a
    left join bluesky_db.main.int_firehose_nlp b
        on a.content_id = b.content_id
    where (first_detected_language = 'English'
        or  first_detected_language is null
        )
    and a.post_created_at_timestamp between to_timestamp_tz('2025-05-25 00:00:00+0000')
                                        and to_timestamp_tz('2025-05-31 23:59:59+0000')
    and b.content_id is null; 
    """
    src_filter = """left join bluesky_db.main.int_firehose_nlp b on a.content_id = b.content_id
    where (first_detected_language = 'English'
        or  first_detected_language is null
        )
    and a.post_created_at_timestamp between to_timestamp_tz('2025-05-25 00:00:00+0000')
                                        and to_timestamp_tz('2025-05-31 23:59:59+0000')
    and b.content_id is null; """
    src_filter = re.sub(r'\s+', ' ', src_filter)
    nlp_params = {'nlp_metric': 'NER_ANALYSIS'
                 ,'transformer_pipeline': PIPL_NER
                 ,'target_text_colname': 'POST_TEXT'
                 }
    target_cols = ['CONTENT_ID', 'POST_CREATED_USA_TIMESTAMP', 'POST_TEXT', 'NER_ANALYSIS', 'SENTIMENT_ANALYSIS']

    # writeback NER
    writeback_batch(query, 'FIREHOSE_PROCESSED', 'INT_FIREHOSE_NLP', target_cols, nlp_params, source_table_filter=src_filter)
    
    ## SENTIMENT ANALYSIS
    query = "select content_id, post_created_usa_timestamp, post_text from bluesky_db.main.int_firehose_nlp;"
    nlp_params = {'nlp_metric': 'SENTIMENT_ANALYSIS'
                 ,'transformer_pipeline': PIPL_SNT
                 ,'target_text_colname': 'POST_TEXT'
                 }
    target_cols = ['POST_TEXT', 'CONTENT_ID', 'SENTIMENT_ANALYSIS']
    
    # writeback SENTIMENT
    writeback_batch(query, 'INT_FIREHOSE_NLP', 'TMP_MERGE_SRC', target_cols, nlp_params)

    # At this point, we have the NER data in INT_FIREHOSE_NLP. We have the SENT data in TMP_MERGE_SRC. So we have to
    # 1. MERGE the SENT data from TMP_MERGE_SRC to INT_FIREHOSE_NLP
    # 2. INSERT everything in INT_FIREHOSE_NLP to FIREHOSE_NLP_LABELED

    query=f"""
    merge into {SF_DB}.{SF_SC}.INT_FIREHOSE_NLP tgt
    using {SF_DB}.{SF_SC}.TMP_MERGE_SRC src
       on src.content_id = tgt.content_id
    when matched then update 
    set tgt.SENTIMENT_ANALYSIS = src.SENTIMENT_ANALYSIS
    """

    CSR = execute_query(query)
    print(f"{(CSR.fetchone()[1]):,} rows updated in INT_FIREHOSE_NLP.SENTIMENT_ANALYSIS, using TMP_MERGE_SRC")

    query = f"""
    insert into {SF_DB}.{SF_SC}.firehose_nlp_labeled
    with src as (
    select a.content_id
          ,a.post_created_usa_timestamp
          ,b.readable_label_name as sentiment_detected_label
          ,cast(sentiment_analysis:score as number(5,4)) as sentiment_confidence_score
          ,row_number() over (
           partition by content_id
           order     by post_created_usa_timestamp, trim(a2.value:word, '"')
           ) as post_entity_number
          ,trim(a2.value:entity_group, '"') as ner_detected_group
          ,trim(a2.value:word, '"') as ner_detected_entity
          ,cast(a2.value:score as number(5,4)) as ner_confidence_score
    from {SF_DB}.{SF_SC}.int_firehose_nlp a
    left join table(flatten(input => parse_json(a.ner_analysis))) a2
    left join {SF_DB}.{SF_SC}.label_map_roberta_base_sentiment b
           on trim(a.sentiment_analysis:label, '"') = b.model_label_name
    )

    select sha2(nvl(to_char(content_id), 'NULL') 
             || '||' 
             || nvl(to_char(post_entity_number), 'NULL')
           ) as analysis_id
          ,*
    from src 
    order by content_id
            ,post_created_usa_timestamp
            ,ner_detected_entity
    """
    try:
        CSR = execute_query(query)
        print(f"{(CSR.fetchone()[0]):,} rows inserted to final target FIREHOSE_NLP_LABELED")
        if CSR.fetchone()[0] > 0:
            CSR = execute_query(f'truncate table {SF_DB}.{SF_SC}.INT_FIREHOSE_NLP')
            print("Successfully cleared INT_FIREHOSE_NLP and inserted all data to FIREHOSE_NLP_LABELED")
    except Exception as e:
        print(f"Encountered an error during the final target insertion: {e}")


Device set to use cuda:0
Some weights of the model checkpoint at dslim/bert-base-NER were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use cuda:0


## STEP 1
Ingest Post-Text into Memory

In [7]:
CSR.execute("select 50")
CSR.fetchone()[0]

50

In [8]:
query = f"""
select content_id
      ,usa_timestamp as post_created_usa_timestamp
      ,post_text
from {SF_DB}.{SF_SC}.firehose_processed
where (first_detected_language = 'English'
   or  first_detected_language is null
   )
  and post_created_usa_timestamp between to_timestamp_tz('2025-05-25 00:00:00+0000')
                                     and to_timestamp_tz('2025-05-31 23:59:59+0000')
;
"""
# and post_created_usa_timestamp >= (select nvl(max(post_created_usa_timestamp), '1900-00-00 00:00:00+0000)
#                                    from {SF_DB}.{SF_SC}.firehose_nlp_labeled) 
#
CSR.execute(query)
total_rows = len(CSR.execute(query).fetch_pandas_all()) # execute once just to get total rows
print(f"{(total_rows):,} downloaded. Processing in batches...\n\n")

# again to start iterating over batches
CSR.execute(query)

c=0
current_pcnt = 0
total_rows_processed = 0
started_at = datetime.now()

print(f"Initiated Named-Entity Recognition (NER) analysis at\n{started_at}")
for batch in CSR.fetch_pandas_batches():
    c+=1
    total_rows_processed += len(batch)
    current_pcnt+=round((len(batch)/total_rows)*100, 1)
    print(f"\n{len(batch):,} rows downloaded from batch {(c):,} ({(current_pcnt):,.1f}% of total rows)")
    
    # using this thing as input is more efficient than using `batch` directly for whatever reason
    batch_dataset = Dataset.from_pandas(batch[['POST_TEXT']], preserve_index=False)
    # execute NER analysis 
    ner_output = PIPL_NER(batch_dataset['POST_TEXT'])
    # Add this col now to match schema-- will populate in the next step
    batch['SENTIMENT_ANALYSIS'] = None
    batch['NER_ANALYSIS']       = ner_output
    # match schema ordinal
    batch = batch[['CONTENT_ID','POST_CREATED_USA_TIMESTAMP','SENTIMENT_ANALYSIS','NER_ANALYSIS','POST_TEXT']]
    
    write_pandas(SF_XCT, batch
            ,table_name = 'INT_FIREHOSE_NLP'
            ,database   = SF_DB.replace('"', '').upper()
            ,schema     = SF_SC.replace('"', '').upper()
            ,use_logical_type = True
            ,auto_create_table = False
            ,overwrite = False
           )
    print(f"{len(batch):,} rows from batch written to {SF_DB}.{SF_SC}.INT_FIREHOSE_NLP")
    min_elapsed = (datetime.now() - started_at).total_seconds/60
    processing_velocity = round(total_rows_processed/min_elapsed, 2)
    print(f"Processing rate = {processing_velocity} rows/minute")

562,204 downloaded. Processing in batches...


Initiated Named-Entity Recognition (NER) analysis at
2025-06-05 17:04:23.526687

572 rows downloaded from batch 1 (0.1% of total rows)
572 rows from batch written to bluesky_db.main.INT_FIREHOSE_NLP
Processing rate = 0.4 rows/minute

1,677 rows downloaded from batch 2 (0.4% of total rows)
1,677 rows from batch written to bluesky_db.main.INT_FIREHOSE_NLP
Processing rate = 1.56 rows/minute

2,941 rows downloaded from batch 3 (0.9% of total rows)


You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


2,941 rows from batch written to bluesky_db.main.INT_FIREHOSE_NLP
Processing rate = 3.61 rows/minute

4,576 rows downloaded from batch 4 (1.7% of total rows)
4,576 rows from batch written to bluesky_db.main.INT_FIREHOSE_NLP
Processing rate = 6.8 rows/minute

11,584 rows downloaded from batch 5 (3.8% of total rows)
11,584 rows from batch written to bluesky_db.main.INT_FIREHOSE_NLP
Processing rate = 14.92 rows/minute

17,697 rows downloaded from batch 6 (6.9% of total rows)
17,697 rows from batch written to bluesky_db.main.INT_FIREHOSE_NLP
Processing rate = 27.45 rows/minute

34,856 rows downloaded from batch 7 (13.1% of total rows)
34,856 rows from batch written to bluesky_db.main.INT_FIREHOSE_NLP
Processing rate = 53.17 rows/minute

216 rows downloaded from batch 8 (13.1% of total rows)
216 rows from batch written to bluesky_db.main.INT_FIREHOSE_NLP
Processing rate = 53.34 rows/minute

517 rows downloaded from batch 9 (13.2% of total rows)
517 rows from batch written to bluesky_db.main

ForbiddenError: 000403: HTTP 403: Forbidden

## STEP 2

Apply Named-Entity Recognition (NER) and write to a stash table (`INT_FIREHOSE_NLP`)

In [ ]:
ner_output = PIPL_NER(DATA['POST_TEXT'].tolist())

# Add this col now to match schema-- will populate in the next step
DATA['SENTIMENT_ANALYSIS'] = None
DATA['NER_ANALYSIS']       = ner_output

write_pandas(SF_XCT, DATA
            ,table_name = 'INT_FIREHOSE_NLP'
            ,database   = SF_DB.replace('"', '').upper()
            ,schema     = SF_SC.replace('"', '').upper()
           )
print(f"{len(DATA):,} rows written to {SF_DB}.{SF_SC}.INT_FIREHOSE_NLP")

KeyboardInterrupt: 

## STEP 3
Re-read data and apply Sentiment analysis, then writeback to stash

In [ ]:
query = f"select POST_TEXT,CONTENT_ID from {SF_DB}.{SF_SC}.int_firehose_nlp"
CSR.execute(query)
DATA = CSR.fetch_pandas_all()
print(f"{len(DATA):,} rows downloaded from INT_FIREHOSE_NLP")

sentiment_output = PIPL_SNT(DATA['POST_TEXT'].tolist())
DATA['SENTIMENT_ANALYSIS'] = sentiment_output

query=f"""
create temp table if not exists {SF_DB}.{SF_SC}.TMP_MERGE_SRC (
 POST_TEXT VARCHAR
,CONTENT_ID VARCHAR
,SENTIMENT_ANALYSIS VARIANT
)"""
CSR.execute(query)

write_pandas(SF_XCT, DATA
            ,table_name = 'TMP_MERGE_SRC'
            ,database   = SF_DB.replace('"', '').upper()
            ,schema     = SF_SC.replace('"', '').upper()
           )
print(f"{len(DATA):,} rows written to {SF_DB}.{SF_SC}.TMP_MERGE_SRC")

query=f"""
merge into {SF_DB}.{SF_SC}.INT_FIREHOSE_NLP tgt
using {SF_DB}.{SF_SC}.TMP_MERGE_SRC src
   on src.content_id = tgt.content_id
when matched then update 
set tgt.SENTIMENT_ANALYSIS = src.SENTIMENT_ANALYSIS
"""
print(f"{(CSR.fetchone()[1]):,} rows updated in INT_FIREHOSE_NLP.SENTIMENT_ANALYSIS, using TMP_MERGE_SRC")

## STEP 4
 Retrieve stashed data, blow it out into many processed rows per-post, then insert.

In [ ]:
query = f"select * from {SF_DB}.{SF_SC}.INT_FIREHOSE_NLP"
CSR.execute(query)
DATA = CSR.fetch_pandas_all()
print(f"{len(DATA):,} rows downloaded from INT_FIREHOSE_NLP")

query = f"""
insert into {SF_DB}.{SF_SC}.firehose_nlp_labeled
with src as (
select a.content_id
      ,a.post_created_usa_timestamp
      ,b.readable_label_name as sentiment_detected_label
      ,cast(sentiment_analysis:score as number(5,4)) as sentiment_confidence_score
      ,row_number() over (
       partition by content_id
       order     by post_created_usa_timestamp, trim(a2.value:word, '"')
       ) as post_entity_number
      ,trim(a2.value:entity_group, '"') as ner_detected_group
      ,trim(a2.value:word, '"') as ner_detected_entity
      ,cast(a2.value:score as number(5,4)) as ner_confidence_score
from {SF_DB}.{SF_SC}.int_firehose_nlp a
left join table(flatten(input => parse_json(a.ner_analysis))) a2
left join {SF_DB}.{SF_SC}.label_map_roberta_base_sentiment b
       on trim(a.sentiment_analysis:label, '"') = b.model_label_name
)

select sha2(nvl(to_char(content_id), 'NULL') 
         || '||' 
         || nvl(to_char(post_entity_number), 'NULL')
       ) as analysis_id
      ,*
from src 
order by content_id
        ,post_created_usa_timestamp
        ,ner_detected_entity
"""
CSR.execute(query)

## Closing
Clear the stash, shut off the Snowflake Connection

In [ ]:
query = f'truncate table {SF_DB}.{SF_SC}.INT_FIREHOSE_NLP'
CSR.execute(query)
SF_XCT.close()

# Unimplemented Code-Corral